# 03_mlflow - Track Ingestion Run with MLflow

This notebook should be run **after** the ingestion notebook has finished writing to the raw table.

It logs:
- Parameters (config path, test folder, run mode, etc.)
- Final metrics (rows written, table stats)
- Artifacts (YAML config, DQ report JSON, optional data sample)

Goal: Demonstrate traceability in CI/CD

In [0]:
import mlflow
from datetime import datetime
import json
import tempfile
import yaml
from pyspark.sql.functions import min, max, countDistinct

In [0]:
dbutils.widgets.text("config_path", "", "YAML config path")
dbutils.widgets.dropdown(
    "test_folder",
    "good",
    ["good", "few_rows", "required_nulls", "high_null_close", "all"],
    "Test subfolder used"
)
dbutils.widgets.text("triggered_by", "manual", "Triggered by (manual / github-actions / etc.)")
dbutils.widgets.text("run_id", "", "Optional: existing MLflow run ID to continue")

config_path = dbutils.widgets.get("config_path").strip()
test_folder = dbutils.widgets.get("test_folder").strip()
triggered_by = dbutils.widgets.get("triggered_by").strip()
existing_run_id = dbutils.widgets.get("run_id").strip()

if not config_path:
    raise ValueError("Please provide config_path (same as used in ingestion notebook)")

In [0]:
cfg_text = dbutils.fs.head(config_path, 200000)
cfg = yaml.safe_load(cfg_text)

raw_table = cfg.get("raw_table")
landing_path = cfg.get("landing_path")
pipeline_name = cfg.get("pipeline_name", "unknown")
source_name = cfg.get("source_name", "unknown")
dataset_name = cfg.get("dataset_name", "unknown")

print("Loaded config:")
print(f"  raw_table     : {raw_table}")
print(f"  landing_path  : {landing_path}")
print(f"  pipeline_name : {pipeline_name}")

In [0]:
experiment_name = "/Shared/LSEG-Ingestion-Experiments"
mlflow.set_experiment(experiment_name)

run_name = f"Ingestion-PostProcess-{test_folder}-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

if existing_run_id:
    mlflow.start_run(run_id=existing_run_id, run_name=run_name)
else:
    mlflow.start_run(run_name=run_name)

print(f"MLflow run: {mlflow.active_run().info.run_id}")

In [0]:
tags = dbutils.notebook.getContext().tags()

github_run_id = tags["jobId"] if "jobId" in tags else "manual"
mlflow.log_param("github_run_id", github_run_id)

git_branch = tags["branchName"] if "branchName" in tags else "local"
mlflow.log_param("git_branch", git_branch)

git_commit = tags["commitId"] if "commitId" in tags else "local"
mlflow.log_param("git_commit", git_commit)

# Log as parameters (appear in table view)
mlflow.log_param("environment",        "dev")           # ← HARDCODE or from variable
mlflow.log_param("bundle_target",      "dev-ci")        # your target name

mlflow.log_param("config_path",     config_path)
mlflow.log_param("test_folder",     test_folder)
mlflow.log_param("triggered_by",    triggered_by)
mlflow.log_param("pipeline_name",   pipeline_name)
mlflow.log_param("source_name",     source_name)
mlflow.log_param("dataset_name",    dataset_name)
mlflow.log_param("raw_table",       raw_table)
mlflow.log_param("run_start_time",  datetime.now().isoformat())

In [0]:
try:
    row_count = spark.table(raw_table).count()
    mlflow.log_metric("rows_written", row_count)
    print(f"Rows in {raw_table}: {row_count:,}")

    # Basic statistics
    stats = spark.table(raw_table).agg(
        min("trade_date").alias("min_trade_date"),
        max("trade_date").alias("max_trade_date"),
        countDistinct("isin").alias("distinct_isins")
    ).collect()[0]

    mlflow.log_metric("distinct_isins", stats["distinct_isins"])
    mlflow.log_param("min_trade_date", str(stats["min_trade_date"]))
    mlflow.log_param("max_trade_date", str(stats["max_trade_date"]))

except Exception as e:
    print(f"Could not read final table stats: {e}")
    mlflow.log_metric("rows_written", -1)
    mlflow.log_param("table_read_error", str(e))

In [0]:
# 1. The YAML config file
with tempfile.NamedTemporaryFile(mode="w", suffix=".yml", delete=False) as tmp:
    tmp.write(cfg_text)
    tmp_path = tmp.name

mlflow.log_artifact(tmp_path, artifact_path="configs")
print("Logged config YAML as artifact")

# 2. Optional: small data sample (first 100 rows)
try:
    sample_df = spark.table(raw_table).limit(100)
    sample_path = "/tmp/sample_data.csv"
    sample_df.write.mode("overwrite").option("header", "true").csv(sample_path)

    # Find the actual CSV part file
    part_files = [f.path for f in dbutils.fs.ls(sample_path) if f.name.startswith("part-") and f.name.endswith(".csv")]
    if part_files:
        mlflow.log_artifact(part_files[0], artifact_path="data_samples/sample_100_rows.csv")
        print("Logged 100-row sample as artifact")

except Exception as e:
    print(f"Could not log data sample: {e}")

In [0]:
mlflow.set_tag("project",      "LSEG")
mlflow.set_tag("env",           "dev")           # short key for quick filter
mlflow.set_tag("stage",         "raw-ingestion")
mlflow.set_tag("test_folder",   test_folder)
mlflow.set_tag("trigger",       triggered_by)

print("MLflow logging complete.")
print(f"Run ID: {mlflow.active_run().info.run_id}")